# 04 — ML Model: Weapon Crime Prediction
**Prerequisite:** `02_silver_lapd_crimes.ipynb` must have run.

**Goal:** Predict `Has_Weapon` (binary) using features from **both**:
- Original crime data (Area, Hour, Month, Crm_Cd, Premise_Cd, Vict_Age, Vict_Sex, Vict_Descent)
- NIBRS enrichment (NIBR_Group, Crime_Against, DomesticViolence, HateCrime, GangRelated, Victim_Type)

**Pipeline:**
1. Feature engineering (SQL-based encoding — avoids StringIndexer whitelist issue)
2. Random Forest + Logistic Regression (direct fit — avoids CrossValidator cache overflow)
3. Evaluation: ROC-AUC, Accuracy, F1
4. Feature importance visualisation
5. Save best model

**Serverless fixes applied:**
- `dense_rank()` replaces `StringIndexer` (Py4J whitelist block)
- Direct model fit replaces `CrossValidator` (ML cache overflow on serverless)
- `del model` between fits to stay within cache limits

## 1. Imports

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/raw_data/sparkml_temp"

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import gc

spark.conf.set("spark.ml.connect.modelCacheSize", "2g")

SILVER_TBL = "silver_lapd_crimes"
MODEL_PATH = "/Volumes/workspace/default/raw_data/models/weapon_rf_model"
print("Imports OK")

## 2. Load Silver Table & Prepare Dataset

In [0]:
df = spark.table(SILVER_TBL)
print(f"Silver rows : {df.count():,}")

model_df = (
    df
    .filter(F.col("Has_Weapon").isNotNull())
    .filter(F.col("AREA").isNotNull())
    .filter(F.col("Hour").isNotNull())
    .filter(F.col("Month").isNotNull())
    .select(
        F.col("Has_Weapon").cast(IntegerType()).alias("label"),
        F.col("AREA").cast(DoubleType()),
        F.col("AREA_NAME"),
        F.col("Hour").cast(DoubleType()),
        F.col("Month").cast(DoubleType()),
        F.col("IsWeekend").cast(DoubleType()),
        F.col("Reporting_Delay").cast(DoubleType()),
        F.col("Vict_Age").cast(DoubleType()),
        F.col("Vict_Sex"),
        F.col("Vict_Descent"),
        F.col("Premise_Desc"),
        F.col("Crm_Cd_Desc"),
        F.col("Part_1_2").cast(DoubleType()),
        F.col("NIBR_Group"),
        F.col("Crime_Against"),
        F.col("DomesticViolence").cast(DoubleType()),
        F.col("HateCrime").cast(DoubleType()),
        F.col("GangRelated").cast(DoubleType()),
        F.col("HomelessVictim").cast(DoubleType()),
        F.col("Victim_Type"),
    )
    .fillna("Unknown", subset=["Vict_Sex", "Vict_Descent", "Premise_Desc",
                                "Crm_Cd_Desc", "NIBR_Group", "Crime_Against",
                                "Victim_Type", "AREA_NAME"])
    .fillna(0.0, subset=["Vict_Age", "IsWeekend", "Reporting_Delay", "Part_1_2",
                          "DomesticViolence", "HateCrime", "GangRelated",
                          "HomelessVictim", "AREA", "Hour", "Month"])
)

pos = model_df.filter(F.col("label") == 1).count()
neg = model_df.filter(F.col("label") == 0).count()
total = model_df.count()
print(f"\nModel dataset   : {total:,} rows")
print(f"  Weapon (1)    : {pos:,}  ({pos/total*100:.1f}%)")
print(f"  No Weapon (0) : {neg:,}  ({neg/total*100:.1f}%)")

## 3. Categorical Encoding via SQL

Uses `dense_rank()` instead of `StringIndexer` (blocked on Shared/Serverless clusters).

In [0]:
cat_cols = [
    "AREA_NAME", "Vict_Sex", "Vict_Descent",
    "Premise_Desc", "Crm_Cd_Desc",
    "NIBR_Group", "Crime_Against", "Victim_Type"
]

encoded_df = model_df
for col_name in cat_cols:
    freq_df = (
        model_df.groupBy(col_name)
        .count()
        .withColumn("_rank", F.dense_rank().over(
            Window.orderBy(F.desc("count"))
        ) - 1)
        .select(
            F.col(col_name).alias(f"_join_{col_name}"),
            F.col("_rank").cast(DoubleType()).alias(f"{col_name}_idx")
        )
    )
    encoded_df = (
        encoded_df
        .join(freq_df,
              encoded_df[col_name] == freq_df[f"_join_{col_name}"],
              "left")
        .drop(f"_join_{col_name}")
    )

idx_cols = [f"{c}_idx" for c in cat_cols]
encoded_df = encoded_df.fillna(0.0, subset=idx_cols)

print(f"Encoded {len(cat_cols)} categorical columns")
print(f"Encoded dataset: {encoded_df.count():,} rows, {len(encoded_df.columns)} cols")

## 4. Train / Test Split

In [0]:
train_df, test_df = encoded_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {train_df.count():,}  |  Test : {test_df.count():,}")

## 5. Feature Assembly

In [0]:
num_cols = [
    "AREA", "Hour", "Month", "IsWeekend",
    "Reporting_Delay", "Vict_Age", "Part_1_2",
    "DomesticViolence", "HateCrime", "GangRelated", "HomelessVictim"
]

feature_cols = idx_cols + num_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features"
)

print(f"Total feature columns: {len(feature_cols)}")
print(f"  Categorical (encoded): {len(idx_cols)}")
print(f"  Numeric: {len(num_cols)}")

## 6. Random Forest

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42,
    numTrees=100,
    maxDepth=10
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

print("Fitting Random Forest...")
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)
print("Done.")

In [0]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

rf_roc = evaluator_roc.evaluate(rf_predictions)
rf_acc = evaluator_acc.evaluate(rf_predictions)
rf_f1  = evaluator_f1.evaluate(rf_predictions)

print(f"Random Forest  ROC-AUC  : {rf_roc:.4f}")
print(f"Random Forest  Accuracy : {rf_acc:.4f}")
print(f"Random Forest  F1-Score : {rf_f1:.4f}")

## 7. Feature Importance

In [0]:
best_rf = rf_model.stages[-1]
importances = best_rf.featureImportances.toArray()
feat_names = [f"{c}_idx" for c in cat_cols] + num_cols

def feat_color(name):
    nibrs_feats = ["NIBR_Group_idx", "Crime_Against_idx", "Victim_Type_idx",
                   "DomesticViolence", "HateCrime", "GangRelated", "HomelessVictim"]
    return "coral" if name in nibrs_feats else "steelblue"

colors = [feat_color(f) for f in feat_names]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sorted_idx = np.argsort(importances)
axes[0].barh([feat_names[i] for i in sorted_idx],
             importances[sorted_idx],
             color=[colors[i] for i in sorted_idx],
             edgecolor="black")
axes[0].set_xlabel("Importance")
axes[0].set_title("Random Forest — All Feature Importances")
axes[0].grid(True, alpha=0.3, axis="x")
base_patch = mpatches.Patch(color="steelblue", label="Crime-base features")
nibr_patch = mpatches.Patch(color="coral",     label="NIBRS features")
axes[0].legend(handles=[base_patch, nibr_patch])

top10_idx = np.argsort(importances)[-10:]
axes[1].barh([feat_names[i] for i in top10_idx],
             importances[top10_idx],
             color=[colors[i] for i in top10_idx],
             edgecolor="black")
for i, (idx, val) in enumerate(zip(top10_idx, importances[top10_idx])):
    axes[1].text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=9)
axes[1].set_xlabel("Importance")
axes[1].set_title("Top 10 Most Important Features")
axes[1].grid(True, alpha=0.3, axis="x")
axes[1].legend(handles=[base_patch, nibr_patch])

plt.suptitle("Feature Importance — Has_Weapon Prediction",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/model_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save RF Model & Free Memory

In [0]:
rf_results = {"roc": rf_roc, "acc": rf_acc, "f1": rf_f1}

rf_model.write().overwrite().save(MODEL_PATH)
print(f"✓ Model saved to: {MODEL_PATH}")

del rf_model, rf_predictions
gc.collect()
print("✓ RF model freed from memory")

## 9. Logistic Regression (Comparison)

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01,
    family="binomial"
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])

print("Fitting Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

lr_roc = evaluator_roc.evaluate(lr_predictions)
lr_acc = evaluator_acc.evaluate(lr_predictions)
lr_f1  = evaluator_f1.evaluate(lr_predictions)

print(f"Logistic Regression  ROC-AUC  : {lr_roc:.4f}")
print(f"Logistic Regression  Accuracy : {lr_acc:.4f}")
print(f"Logistic Regression  F1-Score : {lr_f1:.4f}")

## 10. Visualisations

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar comparison
models = ["Random\nForest", "Logistic\nRegression"]
roc_vals = [rf_results["roc"], lr_roc]
acc_vals = [rf_results["acc"], lr_acc]
f1_vals  = [rf_results["f1"],  lr_f1]
x = np.arange(len(models))
w = 0.25
axes[0].bar(x - w, roc_vals, w, label="ROC-AUC",  color="steelblue",  edgecolor="black")
axes[0].bar(x,     acc_vals, w, label="Accuracy",  color="darkorange", edgecolor="black")
axes[0].bar(x + w, f1_vals,  w, label="F1-Score",  color="mediumseagreen", edgecolor="black")
axes[0].set_xticks(x); axes[0].set_xticklabels(models)
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel("Score")
axes[0].set_title("Model Comparison — All Metrics")
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis="y")

# Confusion matrix (LR)
cm_data = (
    lr_predictions
    .groupBy("label", "prediction")
    .count()
    .toPandas()
)
cm = np.zeros((2, 2))
for _, row in cm_data.iterrows():
    cm[int(row["label"])][int(row["prediction"])] = row["count"]

im = axes[1].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(["Pred: No Weapon", "Pred: Weapon"])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(["True: No Weapon", "True: Weapon"])
axes[1].set_title("Logistic Regression — Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{int(cm[i,j]):,}",
                ha="center", va="center", fontsize=13,
                color="white" if cm[i,j] > cm.max()*0.5 else "black")

plt.suptitle("Model Evaluation — Weapon Crime Prediction",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Results Summary

In [0]:
print("=" * 65)
print(f"{'Model':<25} {'ROC-AUC':>10} {'Accuracy':>10} {'F1':>10}")
print("-" * 65)
print(f"{'Random Forest':<25} {rf_results['roc']:>10.4f} {rf_results['acc']:>10.4f} {rf_results['f1']:>10.4f}")
print(f"{'Logistic Regression':<25} {lr_roc:>10.4f} {lr_acc:>10.4f} {lr_f1:>10.4f}")
print("=" * 65)

rows = [
    ("Random Forest",       rf_results["roc"], rf_results["acc"], rf_results["f1"]),
    ("Logistic Regression", float(lr_roc),     float(lr_acc),     float(lr_f1)),
]
display(spark.createDataFrame(rows, ["Model", "ROC_AUC", "Accuracy", "F1_Score"]))

---
## Summary

| Item | Detail |
|---|---|
| **Target** | `Has_Weapon` (binary: 0 = no weapon, 1 = weapon used) |
| **Features from crime base** | AREA, Hour, Month, IsWeekend, Reporting_Delay, Vict_Age, Vict_Sex, Vict_Descent, Premise_Desc, Crm_Cd_Desc |
| **Features from NIBRS** | NIBR_Group, Crime_Against, DomesticViolence, HateCrime, GangRelated, HomelessVictim, Victim_Type |
| **Encoding** | SQL `dense_rank()` by frequency (avoids StringIndexer Py4J whitelist issue) |
| **Models** | Random Forest (100 trees, depth=10) + Logistic Regression (regParam=0.01) |
| **Saved to** | `/Volumes/workspace/default/raw_data/models/weapon_rf_model` |